# Diabetic Retinopathy Detection — Colab GPU Training

Trains the DR grading model on APTOS 2019 using Colab's free GPU.

**Before running:** Runtime → Change runtime type → **GPU (T4)**.

> Colab already ships a CUDA build of PyTorch. We install only the *extra*
> packages — installing torch here would replace the GPU build with a CPU wheel.

## 1. Confirm the GPU is active

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Set Runtime > Change runtime type > GPU'
print('GPU:', torch.cuda.get_device_name(0))

## 2. Get the project code

**Option A — clone from GitHub (recommended):**

In [ ]:
!git clone https://github.com/johnpitteera/blindness-eradication.git
%cd blindness-eradication

**Option B — upload a zip of the project** (no GitHub needed):

In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()                      # choose your project .zip
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall('.')
# %cd into the extracted folder if needed, e.g. %cd 'Blindness Eradication'

## 3. Install ONLY the extras (leave Colab's torch alone)

In [ ]:
!pip install -q timm albumentations grad-cam pyyaml scikit-learn
print('extras installed; torch untouched')

## 4. Download APTOS 2019 via Kaggle

Add your Kaggle key in Colab's 🔑 **Secrets** panel as `KAGGLE_USERNAME` and
`KAGGLE_KEY`, then run this. (You must accept the competition rules once on the
Kaggle website first — see `data/README.md`.)

In [ ]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

!pip install -q kaggle
!mkdir -p data/aptos2019
!kaggle competitions download -c aptos2019-blindness-detection -p data/aptos2019
!cd data/aptos2019 && unzip -oq aptos2019-blindness-detection.zip
!ls data/aptos2019 | head

## 5. (Optional) Verify preprocessing on one image

In [ ]:
import pandas as pd, cv2, matplotlib.pyplot as plt
from src.preprocessing import preprocess_image

df = pd.read_csv('data/aptos2019/train.csv')
sample_id = df.iloc[0]['id_code']
bgr = cv2.imread(f'data/aptos2019/train_images/{sample_id}.png')
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
proc = preprocess_image(rgb, image_size=512)
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(rgb); ax[0].set_title('original'); ax[0].axis('off')
ax[1].imshow(proc); ax[1].set_title('preprocessed'); ax[1].axis('off')
plt.show()

## 6. Train

Edit `config.yaml` for epochs/batch size. Lower `training.batch_size` to 8 if
you hit CUDA OOM at 512px. Checkpoints + metrics land in `outputs/`.

**First-run tip:** a full 20-epoch run at 512px on a free T4 takes roughly
1.5–2 hours and may approach Colab's idle/session limits. For a first end-to-end
pass, set `training.epochs: 5` (and optionally `preprocessing.image_size: 384`)
to get a working checkpoint in ~25–30 min, then scale up once it's proven.
Early stopping (`early_stop_patience`) will also cut runs short automatically
once validation QWK plateaus.

In [ ]:
!python -m src.train --config config.yaml

## 7. Inspect results

In [ ]:
import json
from IPython.display import Image, display
metrics = json.load(open('outputs/metrics.json'))
print('Best QWK:', metrics['best_qwk'])
print('Referable sensitivity:', metrics['final']['referable_sensitivity'])
display(Image('outputs/confusion.png'))

## 8. Predict + explain a single image

In [ ]:
img = f'data/aptos2019/train_images/{sample_id}.png'
!python -m src.inference --checkpoint outputs/best_model.pth --image {img}
!python -m src.gradcam --checkpoint outputs/best_model.pth --image {img} --output outputs/cam.png
from IPython.display import Image, display
display(Image('outputs/cam.png'))

## 9. Save the model to Drive (so it survives the session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp outputs/best_model.pth '/content/drive/MyDrive/dr_best_model.pth'
print('saved to Drive')